In [11]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent.resolve()))

from planning import (
    air_cargo,
    PlanningProblem,
    Action,
    GraphPlan,
    PartialOrderPlanner,
    Linearize,
    linearize,
    ForwardPlan,
    BackwardPlan
)

from search import Problem, astar_search
from utils import Expr

## Search using Planner



### A* Process with ForwardPlan

**A* Search** is a generic graph exploration algorithm that treats planning as a state-space search problem. **ForwardPlan** implements the `Problem` interface to make planning work with A*.

### Core Mechanism

1. **ForwardPlan wraps a planning problem** as a searchable state space
2. **A* initializes** with ForwardPlan's initial state
3. **A* repeatedly expands nodes** by calling ForwardPlan's methods:
   - `actions(state)` → applicable planning actions
   - `result(state, action)` → new state after action
   - `goal_test(state)` → check if goal reached
4. **A* uses heuristic** `h(state)` to guide search toward goal
5. **A* returns** solution path as sequence of planning actions



```text
┌─────────────────────────────────────────────────────────────────┐
│                    A* Search Process                            │
│  (from search.py)                                               │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
        ┌──────────────────────────────────────┐
        │  Call: astar_search(problem=plan,    │
        │         h=plan.h, display=True)      │
        │                                      │
        │  - problem: ForwardPlan instance     │
        │  - h: heuristic function             │
        └──────────────────────────────────────┘
                              │
                              ▼
        ┌──────────────────────────────────────┐
        │  best_first_graph_search()           │
        │  (Uses f(n) = g(n) + h(n))           │
        └──────────────────────────────────────┘
                              │
        ┌─────────────────────┴────────────────────────────┐
        │                                                  │
        ▼                                                  ▼
┌──────────────────────┐                    ┌──────────────────────┐
│  ForwardPlan Methods │                    │  Search Algorithm    │
│  Called Each Step    │                    │  Manages Frontier    │
│                      │                    │  & Exploration       │
│ - actions(state)     │◄──────────────────►│                      │
│   Returns applicable │                    │ - Expand nodes       │
│   actions for state  │                    │ - Calculate f(n)     │
│                      │                    │ - Track explored     │
│ - result(state,      │◄──────────────────►│                      │
│   action)            │                    │ - Pop min f(n) node  │
│   Returns new state  │                    │ - Check goal         │
│   after action       │                    │                      │
│                      │                    │                      │
│ - goal_test(state)   │◄──────────────────►│                      │
│   Checks if goal     │                    │                      │
│   reached            │                    │                      │
│                      │                    │                      │
│ - h(state)           │◄──────────────────►│ Uses heuristic to    │
│   Heuristic estimate │                    │ guide search         │
│   of remaining cost  │                    │                      │
└──────────────────────┘                    └──────────────────────┘
        │                                                  │
        └─────────────────────┬────────────────────────────┘
                              ▼
                    ┌──────────────────────┐
                    │  Solution Found:     │
                    │  Returns Node with:  │
                    │  - Final state       │
                    │  - Path to solution  │
                    │  - All actions taken │
                    └──────────────────────┘




### Key Insight
- A* doesn't know about planning  
- ForwardPlan doesn't know about A*

They communicate through a simple interface:
- Problem provides: `initial`, `actions()`, `result()`, `goal_test()`, `h()`
- A* uses these to explore the state space efficiently using Node() methods


**Separation of concerns makes the algorithm work for any domain (puzzles, robotics, planning, etc.)**



_example_:  
```text
┌──────────────────────────────────────────────────────────┐
│ STEP 1: Initialize                                       │
│ problem = air_cargo()                                    │
│ plan = ForwardPlan(problem)        ◄─ Wraps problem      │
│ node = astar_search(plan, h=plan.h)                      │
└──────────────────────────────────────────────────────────┘
                         │
                         ▼
┌──────────────────────────────────────────────────────────┐
│ STEP 2: Create Initial Node                              │
│ node = Node(plan.initial)          ◄─ At(C1,SFO) & ...   │
│ node.path_cost = 0                                       │
│ frontier = [node]                                        │
└──────────────────────────────────────────────────────────┘
                         │
                         ▼
┌──────────────────────────────────────────────────────────┐
│ STEP 3: Main Loop (While Frontier Not Empty)             │
│                                                          │
│ node = frontier.pop()  ◄─ Min f(n) node                  │
└──────────────────────────────────────────────────────────┘
                         │
                    ┌────┴────┐
                    │         │
                    ▼         ▼
        ┌─────────────────┐  ┌──────────────────────┐
        │ Goal Test?      │  │ Expand Node          │
        │                 │  │                      │
        │ plan.goal_test()│  │ for action in        │
        │ ├─ NO: expand   │  │   plan.actions(state)│
        │ └─ YES: return  │  │   s' = plan.result() │
        │     solution    │  │   child = Node(s')   │
        └─────────────────┘  └──────────────────────┘
                                    │
                                    ▼
                            ┌──────────────────┐
                            │ Calculate f(n)   │
                            │                  │
                            │ f = 1 +          │
                            │     h(s') = 3    │
                            │ f = 4            │
                            └──────────────────┘
                                    │
                                    ▼
                            ┌──────────────────┐
                            │ Add to frontier  │
                            │ (ordered by f)   │
                            └──────────────────┘
                                    │
                                    └─── LOOP
┌──────────────────────────────────────────────────────────┐
│ STEP 4: Solution Path                                    │
│                                                          │
│ Return: Node with sequence of actions                    │
│ [Load(C1,P1,SFO), Fly(P1,SFO,JFK), Unload(C1,P1,JFK)]    │
└──────────────────────────────────────────────────────────┘

**Detailed Component Interactions**

| Component | Role | Method Called | Returns |
|-----------|------|--------------|---------|
| **A* Algorithm** | Main search loop | `node.expand(problem)` | Child nodes |
| **ForwardPlan** | Planning adapter | `actions(state)` | List of actions `[a1, a2, ...]` |
| **ForwardPlan** | Planning adapter | `result(state, action)` | New state `s'` |
| **ForwardPlan** | Goal checker | `goal_test(state)` | Boolean |
| **ForwardPlan** | Heuristic guide | `h(state)` | Integer cost estimate |
| **Node Class** | Search state | `child_node()` | New `Node(s', parent, action, cost)` |



### ForwardPlan

In [9]:
problem = air_cargo()
plan = ForwardPlan(problem)
node = astar_search(problem=plan, h=plan.h, display=True)

17 paths have been expanded and 54 paths remain in the frontier


In [6]:
if node is None:
    print("No solution")
else:
    actions = node.solution()
    # convert to Expr when possible for nicer printing
    def to_expr(a):
        try:
            return Expr(a.name, *a.args)
        except Exception:
            return a
        
    plan = [to_expr(a) for a in actions]
    print(plan)

[Load(C2, P2, JFK), Fly(P2, JFK, SFO), Unload(C2, P2, SFO), Load(C1, P2, SFO), Fly(P2, SFO, JFK), Unload(C1, P2, JFK)]


### BackwardPlan

In [12]:
problem = air_cargo()
plan = BackwardPlan(problem)
node = astar_search(problem=plan, h=plan.h, display=True)

39 paths have been expanded and 103 paths remain in the frontier


In [13]:
if node is None:
    print("No solution")
else:
    actions = node.solution()
    # convert to Expr when possible for nicer printing
    def to_expr(a):
        try:
            return Expr(a.name, *a.args)
        except Exception:
            return a
        
    plan = [to_expr(a) for a in actions]
    print(plan)

[Unload(C2, P1, SFO), Fly(P1, JFK, SFO), Load(C2, P1, JFK), Unload(C1, P1, JFK), Fly(P1, SFO, JFK), Load(C1, P1, SFO)]


## GraphPlans

### Ground Handling Problem

In [ ]:
def total_ground_handling_problem(n_planes=2):
    """
    Ground handling problem: Service and depart N aircraft at one airport.
    
    Initial: Planes at OnBlock
    Goal: All planes departed
    """

    # Generate object names dynamically
    planes = [f'P{i}' for i in range(1, n_planes + 1)]
    
    # Build initial state (only positive fluents/literals)
    initial_facts = []
    for p in planes:
        initial_facts.append(f'OnBlock({p})')
        #initial_facts.append(f'Plane({p})')
    initial = ' & '.join(initial_facts)
    
    # Build goals
    goals = ' & '.join([f'Departed({p})' for p in planes])

    actions = [
        Action('Unload(p)',
            precond='OnBlock(p)',
            effect='Unloaded(p)',
            domain='Plane(p)'
        ),
        Action('Load(p)',
            precond='Unloaded(p)',
            effect='Loaded(p)',
            domain='Plane(p)'
        ),
        Action('Deboard(p)',
               precond='OnBlock(p)',
               effect='Deboarded(p)',
               domain='Plane(p)'
        ),
        Action('Clean(p)',
            precond='Deboarded(p)',
            effect='Cleaned(p)',
            domain='Plane(p)'
        ),
        Action('Board(p)',
               precond='Cleaned(p)',
               effect='Boarded(p)',
               domain='Plane(p)'
        ),
        Action('Clear(p)',
               precond='Loaded(p) & Boarded(p)',
               effect='AcReady(p)',
               domain='Plane(p)'
        ),
        Action('Push(p)',
               precond='AcReady(p)',
               effect='OffBlock(p) & ~OnBlock(p)',
               domain='Plane(p)'
        ),
        Action(
            'Close(p)',
            precond='OffBlock(p)',
            effect='Departed(p)',
            domain='Plane(p)'
        )
    ]
    
    domain = (
        ' & '.join([f'Plane({p})' for p in planes])
    )

    return PlanningProblem(initial, goals, actions, domain)

### GraphPlan

In [3]:
problem = total_ground_handling_problem(n_planes=2)
planner = GraphPlan(problem)
solution = planner.execute()

# filter in linearize
linearize(solution)


[Deboard(P1),
 Deboard(P2),
 Unload(P1),
 Clean(P2),
 Unload(P2),
 Clean(P1),
 Load(P1),
 Board(P1),
 Board(P2),
 Load(P2),
 Clear(P1),
 Clear(P2),
 Push(P1),
 Push(P2),
 Close(P1),
 Close(P2)]

### Total Order

In [6]:
problem = total_ground_handling_problem(n_planes=2)
planner = Linearize(problem)
planner.execute()

[Deboard(P1),
 Deboard(P2),
 Clean(P1),
 Clean(P2),
 Unload(P1),
 Unload(P2),
 Board(P1),
 Board(P2),
 Load(P1),
 Load(P2),
 Clear(P1),
 Clear(P2),
 Push(P1),
 Push(P2),
 Close(P1),
 Close(P2)]

### Partial Order

In [ ]:

def partial_ground_handling_problem(n_planes=2):
    """
    Ground handling problem: Service and depart N aircraft at one airport.
    
    Initial: Planes at OnBlock
    Goal: All planes departed
    """
    
    # Generate object names dynamically
    planes = [f'P{i}' for i in range(1, n_planes + 1)]
    
    # Build initial state
    initial_facts = []
    for i, p in enumerate(planes):
        initial_facts.append(f'OnBlock({p}) & Unloaded({p}) & ~Loaded({p})')
    
    initial = ' & '.join(initial_facts)
    
    # Build goals
    goals = ' & '.join([f'Departed({p})' for p in planes])
    
    # Define actions (state for all problem sizes)
    actions = [
        Action('Unload(p)',
               precond='OnBlock(p)',
               effect='Unloaded(p)',
               domain='Plane(p)'),

        Action('Load(p)',
               precond='Unloaded(p)',
               effect='Loaded(p)',
               domain='Plane(p)'),

        Action('Deboard(p)',
               precond='OnBlock(p)',
               effect='Deboarded(p)',
               domain='Plane(p)'),

        Action('Board(p)',
               precond='OnBlock(p) & Cleaned(p)',
               effect='Boarded(p)',
               domain='Plane(p)'),

        Action('Clean(p)',
               precond='OnBlock(p) & Deboarded(p)',
               effect='Cleaned(p)',
               domain='Plane(p)'),

        Action('Water(p)',
               precond='OnBlock(p)',
               effect='WaterFilled(p)',
               domain='Plane(p)'),

        Action('Toilet(p)',
               precond='OnBlock(p)',
               effect='ToiletEmpty(p)',
               domain='Plane(p)'),

        Action('Fuel(p)',
               precond='OnBlock(p) & Deboarded(p)',
               effect='Fueled(p)',
               domain='Plane(p)'),

        Action('Clearance(p)',
               precond='OnBlock(p) & Cleaned(p) & Fueled(p) & WaterFilled(p) & ToiletEmpty(p) & Loaded(p) & Boarded(p)',
               effect='AcReady(p)',
               domain='Plane(p)'),

        Action('Pushback(p)',
               precond='OnBlock(p) & AcReady(p)',
               effect='OffBlock(p) & ~OnBlock(p)',
               domain='Plane(p)'),

        Action('Close(p)',
               precond='OffBlock(p)',
               effect='Departed(p)',
               domain='Plane(p)')
    ]
    
    # Build domain
    domain_parts = (
        ' & '.join([f'Plane({p})' for p in planes])
    )
    
    return PlanningProblem(initial=initial, goals=goals, actions=actions, domain=domain_parts)


In [ ]:
# ground_handling_problem

ghp = partial_ground_handling_problem(n_planes=4)

In [ ]:

# partial order planning
pop = PartialOrderPlanner(ghp)
pop.execute()